# Week 12 — Re-run and audit a short research-agent trajectory

**Research task:** Rerun a three-stage research routine and compare its saved events with an earlier record.

**Python introduced:** JSON files, bounded loops, stopping after three calls and field-by-field discrepancy records.

This is the runnable coding component. The full literature-led chapter and slides remain to be developed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session12/session12_replication_studio.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.


In [ ]:
SESSION = "session12"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib.util as setup_importlib
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath("/content/GenAI_Soc2026")
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The course repository root could not be found. Start Jupyter from the "
            "GenAI_Soc2026 folder with: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Choose a route and load an instructor-authored original trajectory

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.


In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # Ollama is the local default
original_trajectory = [
    {"stage":"plan","raw_output":"Search for evidence about replication records."},
    {"stage":"inspect","raw_output":"The source names prompts, models, settings and outputs."},
    {"stage":"write","raw_output":"Replication requires preserving the model-dependent research record."},
]
stages = ["plan", "inspect", "write"]
research_question = "What must be saved to understand a changed LLM-assisted result?"


## Run at most three model calls and preserve every event

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
rerun_trajectory = []
for stage in stages:
    prompt = (
        "Stage: " + stage + ". Research question: " + research_question +
        ". Produce one sentence. Prior events: " + json.dumps(rerun_trajectory)
    )
    messages = [{"role":"user","content":prompt}]
    if ROUTE == "openrouter":
        with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
            response = client.chat.send(model=HOSTED_MODEL,messages=messages,temperature=0,max_tokens=80)
        raw_output = response.choices[0].message.content.strip()
    else:
        response = ollama.chat(model=LOCAL_MODEL,messages=messages,options={"temperature":0,"num_predict":80})
        raw_output = response.message.content.strip()
    event = {"stage":stage,"route":ROUTE,"prompt":prompt,"raw_output":raw_output}
    rerun_trajectory.append(event)
    print("Saved event:", event)

## Write the rerun record to a clearly scoped student-output folder

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
output_dir = ROOT / "student_outputs"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "week12_rerun.json"
output_path.write_text(json.dumps(rerun_trajectory, indent=2))
print("Saved to:", output_path)

## Compare the original and rerun stage by stage

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
discrepancies = []
for original, rerun in zip(original_trajectory, rerun_trajectory):
    same_stage = original["stage"] == rerun["stage"]
    same_output = original["raw_output"] == rerun["raw_output"]
    discrepancies.append({"stage":rerun["stage"],"same_stage":same_stage,"same_output":same_output})
print(discrepancies)

# ONE CHANGE: change temperature from 0 to 0.7 in the selected route and rerun.

## Methodological check

The record shows that temperature changed and whether output changed. It cannot attribute every discrepancy to temperature when model artifacts, providers or runtime templates may also differ.
## Recording

Show the saved three-event rerun, make the temperature change and explain which fields were fixed, which changed and which causal explanation remains unsupported.